SQL is a first-class citizen in Spark — `spark.sql(...)` accepts any ANSI SQL string and passes it through the same Catalyst optimizer as the DataFrame API. Both paths produce identical execution plans. This means you can mix SQL and DataFrame code freely in the same application: register a DataFrame as a view, query it with SQL, take the result back as a DataFrame, and continue chaining.

## SQL and DataFrame — One Optimizer

![Spark SQL Catalyst](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-sql-catalyst.svg)

A DataFrame query and its SQL equivalent compile to the same physical plan. Choose whichever form is more readable for the task.

## Setup — Load All Eight Tables

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("SparkSQL")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

DATA = os.path.join(os.path.dirname(os.path.abspath("10-spark-sql-temporary-views.ipynb")), "data")

customers = spark.read.option("header","true").schema(StructType([
    StructField("customer_id",   StringType(),    False),
    StructField("full_name",     StringType(),    False),
    StructField("email",         StringType(),    True),
    StructField("phone",         StringType(),    True),
    StructField("city",          StringType(),    True),
    StructField("state",         StringType(),    True),
    StructField("date_of_birth", DateType(),      True),
    StructField("kyc_verified",  BooleanType(),   False),
    StructField("created_at",    TimestampType(), False),
])).csv(f"{DATA}/customers")

card_accounts = spark.read.schema(StructType([
    StructField("card_id",      StringType(),       False),
    StructField("customer_id",  StringType(),       False),
    StructField("card_type",    StringType(),       False),
    StructField("network",      StringType(),       True),
    StructField("credit_limit", DecimalType(18,2),  True),
    StructField("issued_on",    DateType(),         False),
    StructField("expiry_date",  DateType(),         False),
    StructField("status",       StringType(),       False),
])).json(f"{DATA}/card_accounts")

card_txns = spark.read.schema(StructType([
    StructField("txn_id",      StringType(),       False),
    StructField("card_id",     StringType(),       False),
    StructField("customer_id", StringType(),       False),
    StructField("amount",      DecimalType(18,2),  False),
    StructField("merchant",    StringType(),       True),
    StructField("category",    StringType(),       True),
    StructField("txn_ts",      TimestampType(),    False),
    StructField("status",      StringType(),       False),
    StructField("is_fraud",    BooleanType(),      False),
])).parquet(f"{DATA}/card_transactions")

loan_accounts = spark.read.option("multiLine","true").schema(StructType([
    StructField("loan_id",       StringType(),       False),
    StructField("customer_id",   StringType(),       False),
    StructField("loan_type",     StringType(),       False),
    StructField("principal",     DecimalType(18,2),  False),
    StructField("interest_rate", DoubleType(),       False),
    StructField("tenure_months", IntegerType(),      False),
    StructField("disbursed_on",  DateType(),         False),
    StructField("status",        StringType(),       False),
])).json(f"{DATA}/loan_accounts")

loan_repayments = spark.read.option("header","true").schema(StructType([
    StructField("repayment_id", StringType(),       False),
    StructField("loan_id",      StringType(),       False),
    StructField("customer_id",  StringType(),       False),
    StructField("due_date",     DateType(),         False),
    StructField("paid_date",    DateType(),         True),
    StructField("emi_amount",   DecimalType(18,2),  False),
    StructField("paid_amount",  DecimalType(18,2),  True),
    StructField("status",       StringType(),       False),
])).csv(f"{DATA}/loan_repayments")

bank_accounts = spark.read.schema(StructType([
    StructField("account_id",   StringType(),       False),
    StructField("customer_id",  StringType(),       False),
    StructField("account_type", StringType(),       False),
    StructField("balance",      DecimalType(18,2),  False),
    StructField("branch_code",  StringType(),       True),
    StructField("ifsc",         StringType(),       True),
    StructField("opened_on",    DateType(),         False),
    StructField("is_active",    BooleanType(),      False),
])).orc(f"{DATA}/bank_accounts")

account_txns = spark.read.schema(StructType([
    StructField("txn_id",        StringType(),       False),
    StructField("account_id",    StringType(),       False),
    StructField("customer_id",   StringType(),       False),
    StructField("txn_type",      StringType(),       False),
    StructField("amount",        DecimalType(18,2),  False),
    StructField("balance_after", DecimalType(18,2),  False),
    StructField("description",   StringType(),       True),
    StructField("txn_ts",        TimestampType(),    False),
])).orc(f"{DATA}/account_transactions")

payments = spark.read.format("delta").load(f"{DATA}/payments")

print("All 8 tables loaded.")

## Creating Temporary Views

`createOrReplaceTempView(name)` registers a DataFrame as a SQL view scoped to the current **SparkSession**. It is visible only within this session and is dropped automatically when the session ends.

In [ ]:
# Register all 8 tables as temp views — one line each
customers.createOrReplaceTempView("customers")
card_accounts.createOrReplaceTempView("card_accounts")
card_txns.createOrReplaceTempView("card_transactions")
loan_accounts.createOrReplaceTempView("loan_accounts")
loan_repayments.createOrReplaceTempView("loan_repayments")
bank_accounts.createOrReplaceTempView("bank_accounts")
account_txns.createOrReplaceTempView("account_transactions")
payments.createOrReplaceTempView("payments")

# Verify via the Catalog API
print("Registered views:")
for t in spark.catalog.listTables():
    print(f"  {t.name:30s}  isTemporary={t.isTemporary}")

## Basic SQL Queries

`spark.sql(...)` returns a **DataFrame** — chain any DataFrame operation on the result or call an action directly.

In [ ]:
# Simple SELECT with WHERE and ORDER BY
spark.sql("""
    SELECT customer_id, full_name, city, state
    FROM   customers
    WHERE  kyc_verified = true
    ORDER  BY full_name
""").show(8, truncate=False)

In [ ]:
# Aggregation — total and average spend per transaction category
spark.sql("""
    SELECT
        category,
        COUNT(*)                          AS num_txns,
        ROUND(SUM(amount), 2)             AS total_spend,
        ROUND(AVG(amount), 2)             AS avg_txn,
        SUM(CAST(is_fraud AS INT))        AS fraud_count
    FROM   card_transactions
    WHERE  status = 'SUCCESS'
    GROUP  BY category
    ORDER  BY total_spend DESC
""").show()

In [ ]:
# HAVING — customers with more than 2 active loans
spark.sql("""
    SELECT
        customer_id,
        COUNT(loan_id)          AS num_loans,
        ROUND(SUM(principal),2) AS total_principal
    FROM   loan_accounts
    WHERE  status = 'ACTIVE'
    GROUP  BY customer_id
    HAVING COUNT(loan_id) > 1
    ORDER  BY total_principal DESC
""").show()

## CTEs — `WITH` Clause

Common Table Expressions (CTEs) make complex queries readable by naming intermediate results. Spark supports full ANSI `WITH ... AS (...)` syntax.

In [ ]:
spark.sql("""
    WITH fraud_summary AS (
        SELECT
            customer_id,
            COUNT(*)              AS fraud_txns,
            ROUND(SUM(amount), 2) AS fraud_amount
        FROM   card_transactions
        WHERE  is_fraud = true
        GROUP  BY customer_id
    ),
    customer_info AS (
        SELECT customer_id, full_name, city
        FROM   customers
        WHERE  kyc_verified = true
    )
    SELECT
        c.customer_id,
        c.full_name,
        c.city,
        f.fraud_txns,
        f.fraud_amount
    FROM   customer_info c
    JOIN   fraud_summary f USING (customer_id)
    ORDER  BY f.fraud_amount DESC
""").show(truncate=False)

## SQL JOINs

Spark SQL supports all standard join types: `INNER`, `LEFT`, `RIGHT`, `FULL OUTER`, `CROSS`, and `SEMI` / `ANTI`.

| Join type | Returns |
|---|---|
| `INNER JOIN` | Rows matching in both tables |
| `LEFT JOIN` | All left rows; nulls for unmatched right |
| `RIGHT JOIN` | All right rows; nulls for unmatched left |
| `FULL OUTER JOIN` | All rows from both; nulls where no match |
| `LEFT SEMI JOIN` | Left rows that have a match in right (no right columns) |
| `LEFT ANTI JOIN` | Left rows that have **no** match in right |

In [ ]:
# INNER JOIN — enrich card transactions with customer city
spark.sql("""
    SELECT
        t.txn_id,
        t.customer_id,
        c.full_name,
        c.city,
        t.amount,
        t.category,
        t.status
    FROM   card_transactions t
    INNER  JOIN customers c USING (customer_id)
    WHERE  t.status = 'SUCCESS'
    ORDER  BY t.amount DESC
    LIMIT  10
""").show(truncate=False)

In [ ]:
# LEFT ANTI JOIN — customers who have never made a payment
spark.sql("""
    SELECT c.customer_id, c.full_name, c.city
    FROM   customers c
    LEFT   ANTI JOIN payments p USING (customer_id)
    ORDER  BY c.full_name
""").show(truncate=False)

In [ ]:
# Multi-table join — customer 360 summary
spark.sql("""
    SELECT
        c.customer_id,
        c.full_name,
        c.city,
        COUNT(DISTINCT ca.card_id)    AS num_cards,
        COUNT(DISTINCT la.loan_id)    AS num_loans,
        COUNT(DISTINCT p.payment_id)  AS num_payments
    FROM       customers       c
    LEFT  JOIN card_accounts   ca USING (customer_id)
    LEFT  JOIN loan_accounts   la USING (customer_id)
    LEFT  JOIN payments        p  USING (customer_id)
    GROUP BY c.customer_id, c.full_name, c.city
    ORDER BY num_payments DESC
    LIMIT 10
""").show(truncate=False)

## Subqueries

Spark SQL supports correlated and uncorrelated subqueries in `WHERE`, `FROM`, and `SELECT` positions.

In [ ]:
# Scalar subquery — transactions above the overall average amount
spark.sql("""
    SELECT txn_id, customer_id, amount, category
    FROM   card_transactions
    WHERE  amount > (SELECT AVG(amount) FROM card_transactions WHERE status = 'SUCCESS')
      AND  status = 'SUCCESS'
    ORDER  BY amount DESC
    LIMIT  10
""").show()

In [ ]:
# IN subquery — payments from customers who have an active loan
spark.sql("""
    SELECT payment_id, customer_id, payment_mode, amount, status
    FROM   payments
    WHERE  customer_id IN (
               SELECT customer_id FROM loan_accounts WHERE status = 'ACTIVE'
           )
    ORDER  BY amount DESC
    LIMIT  10
""").show()

## Window Functions in SQL

SQL window functions use the `OVER (PARTITION BY ... ORDER BY ... ROWS BETWEEN ...)` syntax — identical semantics to the DataFrame Window API.

In [ ]:
# Rank transactions by amount within each customer
spark.sql("""
    SELECT
        customer_id,
        txn_id,
        amount,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY amount DESC) AS rn,
        RANK()        OVER (PARTITION BY customer_id ORDER BY amount DESC) AS rnk,
        ROUND(
            SUM(amount) OVER (PARTITION BY customer_id ORDER BY txn_ts
                              ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2
        ) AS running_spend
    FROM  card_transactions
    WHERE status = 'SUCCESS'
    ORDER BY customer_id, rn
    LIMIT 15
""").show()

In [ ]:
# LAG — compare each repayment to the previous one for the same loan
spark.sql("""
    SELECT
        loan_id,
        due_date,
        emi_amount,
        paid_amount,
        status,
        LAG(status) OVER (PARTITION BY loan_id ORDER BY due_date) AS prev_status
    FROM   loan_repayments
    ORDER  BY loan_id, due_date
""").show(12)

## Global Temporary Views

`createOrReplaceGlobalTempView(name)` registers a view in the `global_temp` database — shared across **all** SparkSessions in the same JVM. Reference it as `global_temp.view_name`.

Use session-scoped views (the default) unless you genuinely need cross-session sharing.

In [ ]:
# Register a global temp view
customers.createOrReplaceGlobalTempView("customers_global")

# Must reference it with the global_temp prefix
spark.sql("SELECT customer_id, full_name FROM global_temp.customers_global LIMIT 3").show()

# Create a second session to demonstrate cross-session visibility
spark2 = SparkSession.builder.appName("Session2").getOrCreate()
spark2.sql("SELECT COUNT(*) AS n FROM global_temp.customers_global").show()

## Catalog API

`spark.catalog` lets you inspect and manage databases, tables, views, and columns programmatically.

In [ ]:
# List all registered views
print("Tables / views in current session:")
for t in spark.catalog.listTables():
    kind = "GLOBAL TEMP" if not t.isTemporary and t.database == "global_temp" else \
           "TEMP" if t.isTemporary else "TABLE"
    print(f"  {t.name:30s}  {kind}")

# Check if a view exists before querying
print("\ncustomers exists:", spark.catalog.tableExists("customers"))
print("unknown  exists:", spark.catalog.tableExists("unknown_table"))

# List columns of a view
print("\nColumns in card_transactions:")
for c in spark.catalog.listColumns("card_transactions"):
    print(f"  {c.name:20s}  {c.dataType}  nullable={c.nullable}")

## DataFrame API vs SQL — Side by Side

The same query written both ways to show they are interchangeable:

In [ ]:
from pyspark.sql.functions import col, count, round as _round, sum as _sum

# DataFrame API
df_result = (
    card_txns
    .filter((col("status") == "SUCCESS") & (col("is_fraud") == False))
    .groupBy("category")
    .agg(
        count("txn_id").alias("txns"),
        _round(_sum("amount"), 2).alias("total"),
    )
    .orderBy(col("total").desc())
)

# Equivalent SQL
sql_result = spark.sql("""
    SELECT   category,
             COUNT(txn_id)        AS txns,
             ROUND(SUM(amount),2) AS total
    FROM     card_transactions
    WHERE    status = 'SUCCESS'
      AND    is_fraud = false
    GROUP BY category
    ORDER BY total DESC
""")

print("DataFrame API result:")
df_result.show()

print("SQL result (identical):")
sql_result.show()

# Confirm the plans are the same
print("DataFrame plan:")
df_result.explain()
print("SQL plan:")
sql_result.explain()

## Summary

- `createOrReplaceTempView(name)` registers a DataFrame as a session-scoped SQL view — automatically dropped when the session ends.
- `spark.sql(query)` returns a DataFrame and goes through the same Catalyst optimizer as the DataFrame API — the execution plans are identical.
- SQL supports `WITH` (CTEs), all standard join types, subqueries in `WHERE` / `FROM` / `SELECT`, and `HAVING`.
- Window functions use `OVER (PARTITION BY ... ORDER BY ... ROWS BETWEEN ...)` — same semantics as the DataFrame `Window` API.
- `createOrReplaceGlobalTempView(name)` shares a view across sessions; reference it as `global_temp.name`.
- `spark.catalog` provides a programmatic API to list and inspect tables, views, and columns.
- Choose SQL or DataFrame API based on readability — mix them freely in the same application.